In [2]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [63]:
import boto3
import requests
from bs4 import BeautifulSoup
from pprint import pformat
from paper_bridge.summarizer.configs import Config
from paper_bridge.summarizer.src import EnvVars, Retriever

In [11]:
def parse_webpage(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for script in soup(["footer", "header", "nav", "script", "style"]):
            script.extract()

        text = soup.get_text()

        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = "\n".join(chunk for chunk in chunks if chunk)

        return text

    except Exception as e:
        return f"Error parsing webpage: {str(e)}"

In [15]:
config = Config.load()
profile_name = EnvVars.AWS_PROFILE_NAME.value

default_boto3_session = boto3.Session(
    region_name=config.resources.default_region_name, profile_name=profile_name
)

retriever = Retriever(config, default_boto3_session)

QUERIES = [
    "주어진 논문의 기술 분야에서 최근 주요 발전 사항은 무엇인가요? 논문 내용:",
    "주어진 논문과 최근 발표된 유사 논문들 간의 핵심 차이점은 무엇인가요? 해당 연구 분야의 동향을 분석하고 얻을 수 있는 통찰을 제시해 주세요. 논문 내용:",
]

URL = "https://transformer-circuits.pub/2025/attribution-graphs/biology.html"
parsed_text = parse_webpage(url)

print("Paper content:\n", parsed_text[:500])

2025-03-31 00:16:42,535 - INFO - Using traversal-based retriever with default options


2025-03-31 00:16:42:INFO:p.s.s.logger   :Using traversal-based retriever with default options
Paper content:
 On the Biology of a Large Language Model
Ã
Transformer Circuits Thread
On the Biology of a Large Language Model
On the Biology of a Large Language Model
We investigate the internal mechanisms used by Claude 3.5 Haiku â Anthropic's lightweight production model â in a variety of contexts, using our circuit tracing methodology.
Authors
Jack Lindseyâ ,
Wes Gurnee*,
Emmanuel Ameisen*,
Brian Chen*,
Adam Pearce*,
Nicholas L. Turner*,
Craig Citro*,
David Abrahams,
Shan Carter,
Basil Hosmer,
Jonath


In [17]:
%%time
answers = []
for query in QUERIES:
    query_text = f"{query} {parsed_text}"
    answer = retriever.query(query_text=query_text)
    answers.append(answer)

2025-03-31 00:18:04:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:04:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:04:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:04:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:04:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10


2025-03-31 00:18:28,589 - INFO - Total execution time: 80.98 seconds (1.35 minutes)


2025-03-31 00:18:28:INFO:p.s.s.logger   :Total execution time: 80.98 seconds (1.35 minutes)
2025-03-31 00:18:47:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:47:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:47:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:47:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.amazonaws.com. Connection pool size: 10
2025-03-31 00:18:47:WARNING:u.connectionpool:Connection pool is full, discarding connection: paper-bridge-dev.cluster-cyq3catzzgsc.us-west-2.neptune.ama

2025-03-31 00:19:48,650 - INFO - Total execution time: 80.06 seconds (1.33 minutes)


2025-03-31 00:19:48:INFO:p.s.s.logger   :Total execution time: 80.06 seconds (1.33 minutes)
CPU times: user 37.8 s, sys: 3 s, total: 40.8 s
Wall time: 2min 41s


In [68]:
print("Response:\n", answers[0]["response"], "\n" + "-" * 100)
print(
    "1st Source Node Text:\n", answers[0]["source_nodes"][0]["text"], "\n" + "-" * 100
)
print(
    "1st Source Node Metadata Source:\n",
    pformat(answers[0]["source_nodes"][0]["metadata"]["source"]),
    "\n" + "-" * 100,
)
print(
    "1st Source Node Metadata - 1st Statement:\n",
    pformat(answers[0]["source_nodes"][0]["metadata"]["topics"][0]["statements"][0]),
)

Response:
 최근 주요 발전 사항은 다음과 같습니다:

1. 병렬 메커니즘과 모듈성: 모델이 다양한 메커니즘을 병렬로 실행하여 협력하거나 경쟁하는 모습이 관찰되었습니다. 

2. 추상화: 모델이 여러 도메인에 걸쳐 매우 일반적인 추상화를 사용하는 것으로 보입니다. 예를 들어 다국어 회로에서 언어에 구애받지 않는 메커니즘이 발견되었습니다.

3. 계획 수립: 시 작성 사례에서 모델이 미래 출력을 위한 계획을 내부적으로 수립하는 모습이 관찰되었습니다.

4. 목표로부터 역방향 작업: 모델이 장기적 목표로부터 역방향으로 작업하여 다음 응답을 결정하는 모습이 관찰되었습니다.

5. 내재된 특성: 보상 모델 편향과 관련된 특성이 Human/Assistant 대화에서 항상 활성화되는 것으로 보아, 이러한 특성이 Assistant 캐릭터에 내재되어 있음을 시사합니다.

전반적으로 모델의 내부 메커니즘이 매우 복잡한 것으로 관찰되었습니다. 
----------------------------------------------------------------------------------------------------
1st Source Node Text:
 {
  "source": "2025-03-25 109 2025-03-30T02:43:42.829998+00:00 2503.18878 I Have Covered All the Bases Here: Interpreting Reasoning Features in\n  Large Language Models via Sparse Autoencoders https://arxiv.org/pdf/2503.18878 ['Andrey Galichin', 'Alexey Dontsov', 'Polina Druzhinina', 'Anton Razzhigaev', 'Oleg Y. Rogov', 'Elena Tutubalina', 'Ivan Oseledets'] 2025-03-24T12:54:26+00:00",
  "topic": "Lar

In [69]:
print("Response:\n", answers[1]["response"], "\n" + "-" * 100)
print(
    "1st Source Node Text:\n", answers[1]["source_nodes"][0]["text"], "\n" + "-" * 100
)
print(
    "1st Source Node Metadata Source:\n",
    pformat(answers[1]["source_nodes"][0]["metadata"]["source"]),
    "\n" + "-" * 100,
)
print(
    "1st Source Node Metadata - 1st Statement:\n",
    pformat(answers[1]["source_nodes"][0]["metadata"]["topics"][0]["statements"][0]),
)

Response:
 이 논문들은 대규모 언어모델의 다양한 측면과 메커니즘을 탐구하고 있습니다. 주요 차이점은 다음과 같습니다:

1. 접근 방식: 일부 논문은 특정 작업이나 현상(예: 다단계 추론, 다국어 처리 등)에 초점을 맞추고 있는 반면, 이 논문은 모델 전반의 다양한 메커니즘을 체계적으로 분석하는 데 중점을 둡니다. 

2. 방법론: 이 논문은 교차층 전사기(cross-layer transcoder)와 귀속 그래프(attribution graph)라는 새로운 방법론을 사용하여 모델의 내부 계산을 시각화하고 해석합니다.

3. 범위: 논문은 시, 산술, 의학 진단, 유해 요청 거부 등 다양한 영역에 걸쳐 모델의 메커니즘을 탐구합니다.

4. 발견: 계획 수립, 역방향 추론, 다국어 통합 회로, 메타인지 회로 등 모델의 복잡하고 정교한 메커니즘을 발견했습니다.

이 연구는 대규모 언어모델의 내부 작동 방식에 대한 통찰력을 제공하며, 모델의 투명성과 해석 가능성을 높이는 데 기여할 수 있습니다. 또한 모델 성능 향상과 안전한 AI 개발을 위한 길잡이 역할을 할 수 있습니다. 
----------------------------------------------------------------------------------------------------
1st Source Node Text:
 {
  "source": "2025-03-03 46 2025-03-30T13:20:33.197716+00:00 2502.18600 Chain of Draft: Thinking Faster by Writing Less https://arxiv.org/pdf/2502.18600 ['Silei Xu', 'Wenhao Xie', 'Lingxiao Zhao', 'Pengcheng He'] 2025-02-25T14:36:06+00:00",
  "topic": "Chain-of-Draft Prompting Research",
  "statements": [
    "Chain-of